# R2-06 · Clasificación y clustering — Profundización (opcional) 🔬

**Formación Pública — Rama R2 · Científico de Datos · Notebook de profundización**

Opcional y de **transferencia**: clasificación con datos **puramente categóricos** — reclamos de
consumidores ante el **SERNAC** (muestra de 20.000 reclamos de 2011, dato histórico). Vas a
convertir categorías en números con **one-hot**, superar una **línea base** y ver cómo el
**desbalance** reaparece cuando el problema tiene 7 clases.

> Cada ejercicio termina en una **celda de chequeo** con ✅ / pista.

In [ ]:
import pandas as pd, numpy as np, os, urllib.request
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, classification_report

CSV = "reclamos.csv"
if not os.path.exists(CSV):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/reclamos.csv", CSV)
rec = pd.read_csv(CSV)
print("reclamos:", rec.shape)
print(rec["resultado"].value_counts())

## 1. Transferencia: ¿el proveedor acogerá el reclamo?

Dominio nuevo: **reclamos de consumidores ante el SERNAC** (muestra de 20.000 reclamos de **2011** — un dato histórico: el SERNAC no publica una base abierta más reciente con este detalle). Todas las variables son **categorías** (mercado, motivo legal…), no números.

Objetivo binario: `acoge = 1` si el resultado fue «PROVEEDOR ACOGE». Primero, la vara mínima: la exactitud de predecir siempre la clase más frecuente.

Un modelo no entiende texto: hay que convertir cada categoría en columnas 0/1. Eso es el **one-hot encoding** (`OneHotEncoder`): una columna por cada valor posible.

### ✍️ Ejercicio 1 — Objetivo, línea base y codificación one-hot

In [ ]:
rec["acoge"] = (rec["resultado"] == "PROVEEDOR ACOGE").astype(int)
CAT = ["mercado", "motivo_legal"]
X_tr, X_te, y_tr, y_te = train_test_split(rec[CAT], rec["acoge"], test_size=0.3,
                                          random_state=42, stratify=rec["acoge"])
linea_base = max(y_te.mean(), 1 - y_te.mean())
enc = OneHotEncoder(handle_unknown="ignore").fit(X_tr)
n_columnas = enc.transform(X_tr).shape[1]
print(f"Acoge: {rec['acoge'].mean():.1%} | línea base: {linea_base:.3f} | columnas one-hot: {n_columnas}")

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert 0.5 < linea_base < 0.55
    assert n_columnas == X_tr["mercado"].nunique() + X_tr["motivo_legal"].nunique(), \
        "One-hot crea una columna por cada valor distinto de cada variable"
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "linea_base = max(y_te.mean(), 1 - y_te.mean()); enc = OneHotEncoder(handle_unknown='ignore').fit(X_tr); enc.transform(X_tr).shape[1].")
    print("   Detalle:", e)

## 2. Clasificar con variables categóricas

Con las categorías ya convertidas en números, el flujo es el de siempre. ¿Supera el modelo a la línea base? ¿Por cuánto?

### ✍️ Ejercicio 2 — Entrena una regresión logística sobre el one-hot

In [ ]:
clf = LogisticRegression(max_iter=2000).fit(enc.transform(X_tr), y_tr)
pred = clf.predict(enc.transform(X_te))
exactitud = accuracy_score(y_te, pred)
print(f"Exactitud: {exactitud:.3f} vs línea base {linea_base:.3f}")

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    assert exactitud > linea_base + 0.05, "El modelo debería superar claramente a la línea base"
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'clf = LogisticRegression(max_iter=2000).fit(enc.transform(X_tr), y_tr); pred = clf.predict(enc.transform(X_te)); accuracy_score(y_te, pred).')
    print("   Detalle:", e)

## 3. Siete resultados: el desbalance vuelve

El resultado real tiene **7 categorías**, algunas muy raras (p. ej. «antecedentes insuficientes»). Con un clasificador multiclase, la exactitud puede verse aceptable mientras las clases raras **nunca** se predicen. El **recall macro** (el promedio simple del recall de cada clase) lo delata.

### ✍️ Ejercicio 3 — Exactitud vs recall macro en el problema de 7 clases

In [ ]:
CAT3 = ["mercado", "categoria_mercado", "motivo_legal"]
X_tr3, X_te3, y_tr3, y_te3 = train_test_split(rec[CAT3], rec["resultado"], test_size=0.3,
                                              random_state=42, stratify=rec["resultado"])
enc3 = OneHotEncoder(handle_unknown="ignore").fit(X_tr3)
clf3 = LogisticRegression(max_iter=3000).fit(enc3.transform(X_tr3), y_tr3)
pred3 = clf3.predict(enc3.transform(X_te3))
exactitud3 = accuracy_score(y_te3, pred3)
recall_macro = recall_score(y_te3, pred3, average="macro", zero_division=0)
print(f"Exactitud: {exactitud3:.2f} | recall macro: {recall_macro:.2f}")
print(classification_report(y_te3, pred3, zero_division=0))

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    assert recall_macro < exactitud3 - 0.1, "El recall macro debería quedar muy por debajo de la exactitud"
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "accuracy_score(y_te3, pred3) y recall_score(y_te3, pred3, average='macro', zero_division=0).")
    print("   Detalle:", e)

---

*Fin de la profundización de R2-06.*

*Fuente: SERNAC, reclamos de consumidores 2011 (muestra de 20.000; dato histórico).*